# Install Requirement Libraries

In [1]:
!pip install spacy
!python -m spacy download en_core_web_sm
!pip install neo4j
!pip install sentence-transformers
!pip install pandas

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typer_slim-0.20.0-py3-none-any.whl.metadata (16 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
  Usin

# Requirement Datasets

In [ ]:
import pandas as pd
from pathlib import Path
import spacy
from tqdm import tqdm

nlp = spacy.load("en_core_web_sm")

DATA_DIR = Path(r"E:\Shreyas_Codez\Sustainable_LLM\TF_CDN-0.1.0\garima2751-TF_CDN-50f44df\Data")

FILES = {
    "IBM_UAV": "cdn_pairs.csv",  
    "open_coss": "open_coss_clean_pairs.csv",
    "pure": "pure_clean_pairs.csv",
    "uav": "uav_clean_pairs.csv",
    "world_vista": "world_vista_clean_pairs.csv",
}

def extract_entities(text):
    doc = nlp(text)

    actors = set()
    actions = set()
    objects = set()
    conditions = set()

    for token in doc:
        if token.dep_ in ["nsubj", "nsubjpass"]:
            actors.add(token.text)

        if token.pos_ == "VERB":
            actions.add(token.lemma_)

        if token.dep_ in ["dobj", "pobj"]:
            objects.add(token.text)

    for sent in doc.sents:
        low = sent.text.lower()
        if "when" in low or "if" in low or "after" in low:
            conditions.add(sent.text.strip())

    return {
        "actors": list(actors),
        "actions": list(actions),
        "objects": list(objects),
        "conditions": list(conditions)
    }


def load_all_datasets():
    all_requirements = []
    seen = set()

    for dataset_label, filename in FILES.items():
        df = pd.read_csv(DATA_DIR / filename)

        for req in df["Text1"].dropna().unique():
            clean_req = req.strip()

            if clean_req not in seen:
                seen.add(clean_req)
                entities = extract_entities(clean_req)

                all_requirements.append({
                    "requirement": clean_req,
                    "source": dataset_label,
                    "actors": entities["actors"],
                    "actions": entities["actions"],
                    "objects": entities["objects"],
                    "conditions": entities["conditions"],
                })

    return pd.DataFrame(all_requirements)


if __name__ == "__main__":
    df = load_all_datasets()
    df.to_csv("unique_requirements_with_entities.csv", index=False)
    print("Created: unique_requirements_with_entities.csv")
    print("Total unique requirements:", len(df))


FileNotFoundError: [Errno 2] No such file or directory: 'E:\\\\Shreyas_Codez\\\\Sustainable_LLM\\\\TF_CDN-0.1.0\\\\garima2751-TF_CDN-50f44df\\\\Data/cdn_pairs.csv'

In [1]:
import pandas as pd
from pathlib import Path
import spacy
from tqdm import tqdm
import time
from codecarbon import EmissionsTracker


nlp = spacy.load("en_core_web_sm")

DATA_DIR = Path(r"E:\Shreyas_Codez\Sustainable_LLM\TF_CDN-0.1.0\garima2751-TF_CDN-50f44df\Data")

FILES = {
    "IBM_UAV": "cdn_pairs.csv",
    "open_coss": "open_coss_clean_pairs.csv",
    "pure": "pure_clean_pairs.csv",
    "uav": "uav_clean_pairs.csv",
    "world_vista": "world_vista_clean_pairs.csv",
}


def extract_entities_original(text):
    doc = nlp(text)

    actors = set()
    actions = set()
    objects = set()
    conditions = set()

    for token in doc:
        if token.dep_ in ["nsubj", "nsubjpass"]:
            actors.add(token.text)

        if token.pos_ == "VERB":
            actions.add(token.lemma_)

        if token.dep_ in ["dobj", "pobj"]:
            objects.add(token.text)

    for sent in doc.sents:
        low = sent.text.lower()
        if "when" in low or "if" in low or "after" in low:
            conditions.add(sent.text.strip())

    return {
        "actors": list(actors),
        "actions": list(actions),
        "objects": list(objects),
        "conditions": list(conditions)
    }


def extract_entities_improved(text):
    doc = nlp(text)

    actors = set()
    actions = set()
    objects = set()
    attributes = set()
    conditions = set()

    for ent in doc.ents:
        if ent.label_ in ["ORG", "PRODUCT"]:
            actors.add(ent.text.lower())

    for chunk in doc.noun_chunks:
        root = chunk.root

        if root.dep_ in ["nsubj", "nsubjpass"]:
            actors.add(chunk.text.lower())

        if root.dep_ in ["dobj", "pobj", "attr"]:
            objects.add(root.lemma_.lower())

            for tok in chunk:
                if tok.dep_ in ["amod", "compound"] and tok != root:
                    attributes.add(tok.lemma_.lower())

    for token in doc:
        if token.pos_ == "VERB":
            actions.add(token.lemma_.lower())

    for sent in doc.sents:
        low = sent.text.lower()
        if any(x in low for x in ["if", "when", "after", "before", "unless"]):
            conditions.add(sent.text.strip())

    return {
        "actors": list(actors),
        "actions": list(actions),
        "objects": list(objects),
        "attributes": list(attributes),
        "conditions": list(conditions)
    }



def collect_unique_requirements():
    seen = set()
    items = []

    for dataset_label, filename in FILES.items():
        df = pd.read_csv(DATA_DIR / filename)

        for req in df["Text1"].dropna().unique():
            req = req.strip()
            if req not in seen:
                seen.add(req)
                items.append((req, dataset_label))

    return items




def run_pipeline(extractor, label, requirements):

    tracker = EmissionsTracker(
        project_name=label,
        output_dir=".",
        log_level="error"
    )

    tracker.start()
    start = time.time()

    rows = []

    for req, source in tqdm(requirements, desc=label):
        entities = extractor(req)

        row = {
            "requirement": req,
            "source": source,
            **entities
        }
        rows.append(row)

    latency = time.time() - start
    emissions = tracker.stop()

    df = pd.DataFrame(rows)

    return df, latency, emissions


if __name__ == "__main__":

    print("Collecting unique requirements...")
    requirements = collect_unique_requirements()
    print("Total unique requirements:", len(requirements))

    print("\nRunning ORIGINAL pipeline...")
    df_original, latency_orig, emissions_orig = run_pipeline(
        extract_entities_original,
        "original_extraction",
        requirements
    )

    df_original.to_csv("original_unique_requirements_with_entities.csv", index=False)

    print("\nRunning IMPROVED pipeline...")
    df_improved, latency_improved, emissions_improved = run_pipeline(
        extract_entities_improved,
        "improved_extraction",
        requirements
    )

    df_improved.to_csv("improved_unique_requirements_with_entities.csv", index=False)


    metrics = pd.DataFrame([
        {
            "pipeline": "original",
            "latency_seconds": latency_orig,
            "emissions_kgCO2": emissions_orig
        },
        {
            "pipeline": "improved",
            "latency_seconds": latency_improved,
            "emissions_kgCO2": emissions_improved
        }
    ])

    metrics.to_csv("entity_extraction_energy_latency.csv", index=False)

    print("\n===================================")
    print("Saved outputs:")
    print(" - original_unique_requirements_with_entities.csv")
    print(" - improved_unique_requirements_with_entities.csv")
    print(" - entity_extraction_energy_latency.csv")
    print("===================================")


[codecarbon WARNING @ 12:44:40] Multiple instances of codecarbon are allowed to run at the same time.


Total unique requirements: 1179

Running ORIGINAL pipeline...


original_extraction: 100%|██████████| 1179/1179 [00:06<00:00, 185.36it/s]



Running IMPROVED pipeline...


improved_extraction: 100%|██████████| 1179/1179 [00:06<00:00, 182.50it/s]



Saved outputs:
 - original_unique_requirements_with_entities.csv
 - improved_unique_requirements_with_entities.csv
 - entity_extraction_energy_latency.csv


# Insertion in Graph Database

## Insertion W/O Source

In [1]:
from neo4j import GraphDatabase
import pandas as pd
import os 
from ast import literal_eval

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASS = os.environ.get("NEO4J_PASS", "Shreyas@1411")
URI = NEO4J_URI
USER = NEO4J_USER   
PASSWORD = NEO4J_PASS

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

def insert_requirement_and_entities(tx, req_data):

    # Create requirement node
    tx.run("""
        MERGE (r:Requirement {text: $text})
        ON CREATE SET r.id = randomUUID(),
                      r.source = $source
    """, text=req_data["requirement"], source=req_data["source"])

    # Helper to insert entity types
    def insert_entities(entities, entity_type, rel_type):
        if not entities or len(entities) == 0:
            return  # 👈 Skip empty lists

        for e in entities:
            if not e or e.strip() == "":
                continue  # Skip blank strings

            tx.run(f"""
                MERGE (ent:Entity {{name: $name, type: $type}})
                WITH ent
                MATCH (r:Requirement {{text: $req}})
                MERGE (r)-[:{rel_type}]->(ent)
            """, name=e, type=entity_type, req=req_data["requirement"])

    # Parse CSV-stored lists safely
    actors = literal_eval(req_data["actors"])
    actions = literal_eval(req_data["actions"])
    objects = literal_eval(req_data["objects"])
    conditions = literal_eval(req_data["conditions"])

    # Insert each entity type only if non-empty
    insert_entities(actors, "Actor", "HAS_ACTOR")
    insert_entities(actions, "Action", "HAS_ACTION")
    insert_entities(objects, "Object", "HAS_OBJECT")
    insert_entities(conditions, "Condition", "HAS_CONDITION")


def load_and_insert():
    df = pd.read_csv("unique_requirements_with_entities.csv")

    with driver.session() as session:
        for _, row in df.iterrows():
            session.execute_write(insert_requirement_and_entities, row)

    print("Inserted", len(df), "requirements with entities into Neo4j (skipped empty entities).")


if __name__ == "__main__":
    load_and_insert()


Inserted 1179 requirements with entities into Neo4j (skipped empty entities).


## Insertion With Graph

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import os
from ast import literal_eval

NEO4J_URI = "neo4j+s://d1df8710.databases.neo4j.io"
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASS = "xJ6GyOib4r5d2uq8pkj7QrqNtYSyQRPR_h1l8HiFvRc"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def insert_requirement_and_entities(tx, req_data):

    # --- Create Requirement (scoped by source) ---
    tx.run("""
        MERGE (r:Requirement {text: $text, source: $source})
        ON CREATE SET r.id = randomUUID()
    """, text=req_data["requirement"], source=req_data["source"])

    # --- Helper to insert entities ---
    def insert_entities(entities, entity_type, rel_type):
        if not entities:
            return

        for e in entities:
            if not e or e.strip() == "":
                continue

            tx.run(f"""
                MERGE (ent:Entity {{
                    name: $name,
                    type: $type,
                    source: $source
                }})
                WITH ent
                MATCH (r:Requirement {{
                    text: $req,
                    source: $source
                }})
                MERGE (r)-[:{rel_type}]->(ent)
            """,
            name=e,
            type=entity_type,
            source=req_data["source"],
            req=req_data["requirement"]
            )

    # --- Parse CSV list fields safely ---
    actors = literal_eval(req_data["actors"])
    actions = literal_eval(req_data["actions"])
    objects = literal_eval(req_data["objects"])
    conditions = literal_eval(req_data["conditions"])

    insert_entities(actors, "Actor", "HAS_ACTOR")
    insert_entities(actions, "Action", "HAS_ACTION")
    insert_entities(objects, "Object", "HAS_OBJECT")
    insert_entities(conditions, "Condition", "HAS_CONDITION")


def load_and_insert():
    df = pd.read_csv("unique_requirements_with_entities.csv")

    with driver.session() as session:
        for _, row in df.iterrows():
            session.execute_write(insert_requirement_and_entities, row)

    print(f"Inserted {len(df)} requirements into Neo4j (source-isolated).")


if __name__ == "__main__":
    load_and_insert()

Inserted 1179 requirements into Neo4j (source-isolated).


## Insertion of Improved KAG

In [1]:
from neo4j import GraphDatabase
import pandas as pd
import os
from ast import literal_eval

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASS = os.environ.get("NEO4J_PASS", "Shreyas@1411")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def insert_requirement_and_entities(tx, req_data):

    tx.run("""
        MERGE (r:Requirement {text: $text, source: $source})
        ON CREATE SET r.id = randomUUID()
    """, text=req_data["requirement"], source=req_data["source"])


    def insert_entities(entities, entity_type, rel_type):
        if not entities:
            return

        for e in entities:
            if not e or str(e).strip() == "":
                continue

            tx.run(f"""
                MERGE (ent:Entity {{
                    name: $name,
                    type: $type,
                    source: $source
                }})
                WITH ent
                MATCH (r:Requirement {{
                    text: $req,
                    source: $source
                }})
                MERGE (r)-[:{rel_type}]->(ent)
            """,
            name=str(e).lower(),
            type=entity_type,
            source=req_data["source"],
            req=req_data["requirement"]
            )


    actors = literal_eval(req_data.get("actors", "[]"))
    actions = literal_eval(req_data.get("actions", "[]"))
    objects = literal_eval(req_data.get("objects", "[]"))
    attributes = literal_eval(req_data.get("attributes", "[]"))   
    conditions = literal_eval(req_data.get("conditions", "[]"))

    insert_entities(actors, "Actor", "HAS_ACTOR")
    insert_entities(actions, "Action", "HAS_ACTION")
    insert_entities(objects, "Object", "HAS_OBJECT")
    insert_entities(attributes, "Attribute", "HAS_ATTRIBUTE")   
    insert_entities(conditions, "Condition", "HAS_CONDITION")


def load_and_insert():
    df = pd.read_csv("improved_unique_requirements_with_entities.csv")

    with driver.session() as session:
        for _, row in df.iterrows():
            session.execute_write(insert_requirement_and_entities, row)

    print(f"Inserted {len(df)} requirements into Neo4j (source-isolated).")


if __name__ == "__main__":
    load_and_insert()

Inserted 1179 requirements into Neo4j (source-isolated).


# Populate Vector DB

In [1]:
!pip install pymilvus

In [2]:
import pandas as pd
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
from sentence_transformers import SentenceTransformer
import uuid
import asyncio # Keep the import, but don't use .run()
from tqdm import tqdm


model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


connections.connect(
  alias="default", 
  host="localhost", 
  port="19530"
)

def create_collection():
    fields = [
        FieldSchema(name="req_id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="uuid", dtype=DataType.VARCHAR, max_length=50),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=2000),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=768),
    ]

    schema = CollectionSchema(fields, description="Requirement Embeddings Collection")

    collection = Collection(
        name="requirements",
        schema=schema,
        using="default",
        shards_num=2
    )
    return collection


async def insert_data(): # Function remains asynchronous (async def)
    df = pd.read_csv("unique_requirements_with_entities.csv")

    # Make sure the collection exists
    try:
        collection = Collection("requirements")
    except:
        collection = create_collection()

    texts = df["requirement"].tolist()
    uuids = [str(uuid.uuid4()) for _ in texts]

    print("\nGenerating embeddings...")
    embeddings = model.encode(texts, batch_size=32, convert_to_numpy=True)

    print("Inserting into Milvus...")
    collection.insert([uuids, texts, embeddings])

    print(f"Inserted {len(texts)} requirements into Milvus.")

    # Create index for fast similarity search
    print("Creating index on embeddings...")
    # NOTE: PyMilvus's create_index is synchronous. No 'await' here.
    collection.create_index( 
        field_name="embedding",
        index_params={
            "index_type": "IVF_FLAT",
            "metric_type": "COSINE",
            "params": {"nlist": 1024}
        }
    )

    collection.load()
    print("Milvus collection loaded and ready.")


if __name__ == "__main__":
    await insert_data()

/Users/shreyaspatil/ShreyasCodez/SustainableLLMs_SystemRequirements/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Generating embeddings...


I0000 00:00:1769089541.735911   26655 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Inserting into Milvus...
Inserted 1179 requirements into Milvus.
Creating index on embeddings...
Milvus collection loaded and ready.
